In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l4.5 Output head and weight tying

The final LayerNorm + linear head projects the hidden state to one logit per
vocab entry. PocketLM **ties** that head to the input embedding: one shared
`[vocab, n_embd]` matrix, fewer parameters, usually better. This completes
PocketLM v1, the model you train for the rest of the course.

In [ ]:
from pocketlm import train_tiny, pick_config, generate

corpus = open(CORPUS_PATH, encoding="utf-8").read()
model, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
print("head tied to embedding:", model.head.weight is model.tok_emb.weight)
print("parameters:", sum(p.numel() for p in model.parameters()))
print(generate(model, tok, "The ", max_new_tokens=40, seed=0))

That is the full architecture: tokenize, embed, add positions, stack blocks,
norm, project to logits. Everything after this unit is about *training* it well
(u5), *modernizing* it (u6), *adapting* it (u7), and *serving* it (u8).

In [ ]:
assert model.head.weight is model.tok_emb.weight
assert len(generate(model, tok, "The ", max_new_tokens=40, seed=0)) == 4 + 40